# Pipeline

## 1. Librerias

In [ ]:
import numpy as np
import scipy.io as sio
import boto3
import json
import random
from io import BytesIO
from datetime import datetime, timedelta
import xgboost as xgb


## 2. Configuración global

In [ ]:
BUCKET = "tfm-mu-bd"
RAW_PREFIX = "raw/synthetic_ecg"
META_PREFIX = "outputs/metadata"

s3 = boto3.client("s3")

EXPECTED_LEADS = 12
EXPECTED_LENGTH = 5000

## 3. Crear archivos sintéticos

In [ ]:
RISK_CONFIG = {
    "BAJO": {"noise": 0.05, "extra_freq": 3, "extra_amp": 0.0, "prob_range": (0.05, 0.49)},
    "MEDIO": {"noise": 0.05, "extra_freq": 3, "extra_amp": 0.15, "prob_range": (0.50, 0.74)},
    "ALTO": {"noise": 0.05, "extra_freq": 5, "extra_amp": 0.4, "prob_range": (0.75, 0.98)}
}

def generate_synthetic_ecg(risk="BAJO"):
    cfg = RISK_CONFIG[risk]

    t = np.linspace(0, 10, EXPECTED_LENGTH)

    base = 0.5 * np.sin(2 * np.pi * 1.2 * t)
    noise = cfg["noise"] * np.random.randn(len(t))
    extra = cfg["extra_amp"] * np.sin(2 * np.pi * cfg["extra_freq"] * t)

    signal = base + noise + extra

    ecg = []
    for _ in range(EXPECTED_LEADS):
        ecg.append(signal + np.random.uniform(-0.2, 0.2))

    return np.array(ecg, dtype=np.float32)



In [ ]:

def upload_synthetic(record_id, ecg):

    buffer = BytesIO()
    sio.savemat(buffer, {"val": ecg})
    buffer.seek(0)

    key = f"{RAW_PREFIX}/{record_id}.mat"

    s3.put_object(
        Bucket=BUCKET,
        Key=key,
        Body=buffer.getvalue()
    )

    return f"s3://{BUCKET}/{key}"


def generate_record(record_id, risk, timestamp):
    cfg = RISK_CONFIG[risk]

    prob = round(random.uniform(*cfg["prob_range"]), 4)

    return {
        "record_id": record_id,
        "probability": prob,
        "risk": risk,
        "timestamp": timestamp.isoformat(),
        "dt": timestamp.date().isoformat()
    }


def create_dataset():

    base_date = datetime.now()

    risks = (["BAJO"] * 70) + (["MEDIO"] * 20) + (["ALTO"] * 10)

    record_id = 100
    total = 700
    created = 0

    while created < total:

        risk = random.choice(risks)

        timestamp = base_date - timedelta(days=random.randint(0, 6))
        timestamp = timestamp.replace(
            hour=random.randint(0, 23),
            minute=random.randint(0, 59),
            second=random.randint(0, 59)
        )

        rid = f"ECG_{record_id:05d}"

        ecg = generate_synthetic_ecg(risk)
        mat_path = upload_synthetic(rid, ecg)

        record = generate_record(rid, risk, timestamp)

        s3.put_object(
            Bucket=BUCKET,
            Key=f"{META_PREFIX}/dt={record['dt']}/{rid}.json",
            Body=json.dumps(record),
            ContentType="application/json"
        )

        record_id += 1
        created += 1


create_dataset()

# PREPROCESADO + FEATURES + INFERENCIA + DASHBOARD OUTPUT 

## 4. Cargar modelo

In [ ]:
local_path = "/tmp/xgboost_model.json"

s3.download_file(
    BUCKET,
    "models/xgboost_model.json",
    local_path
)

xgboost_model = xgb.XGBClassifier()
xgboost_model.load_model(local_path)

## 5. Pipeline

In [ ]:

def load_mat_from_s3(key):
    local = "/tmp/temp.mat"

    s3.download_file(BUCKET, key, local)
    mat = sio.loadmat(local)

    signal = mat["val"]

    if signal.shape[0] != 12:
        signal = signal.T

    return signal.astype(np.float32)

def extract_features(signal):
    return np.concatenate([
        signal.mean(axis=1),
        signal.std(axis=1),
        signal.min(axis=1),
        signal.max(axis=1),
        (signal ** 2).mean(axis=1)
    ])

def build_vector(signal):
    return extract_features(signal)


def model_predict(features):
    features = np.array(features).reshape(1, -1)
    prob = xgboost_model.predict_proba(features)[0, 1]
    return float(prob)


def classify(prob):
    if prob < 0.5:
        return "BAJO"
    elif prob < 0.75:
        return "MEDIO"
    return "ALTO"


def save_prediction(record_id, prob, risk):

    now = datetime.now()

    payload = {
        "record_id": record_id,
        "probability": float(prob),
        "risk": risk,
        "timestamp": now.isoformat(),
        "dt": now.date().isoformat()
    }

    key = f"{PRED_PREFIX}/dt={payload['dt']}/{record_id}_{now.strftime('%Y%m%d_%H%M%S')}.json"

    s3.put_object(
        Bucket=BUCKET,
        Key=key,
        Body=json.dumps(payload),
        ContentType="application/json"
    )

    return f"s3://{BUCKET}/{key}"


def run_pipeline_from_s3(raw_key, record_id):

    signal = load_mat_from_s3(raw_key)

    features = build_vector(signal)

    prob = model_predict(features)

    risk = classify(prob)

    return save_prediction(record_id, prob, risk)